<a href="https://colab.research.google.com/github/csugasi/DATASCI266-Banking77-Encoder-vs-Decoder/blob/shuchi%2Fdev/notebooks/qwen_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os

token = userdata.get('GH_TOKEN')
repo  = "csugasi/DATASCI266-Banking77-Encoder-vs-Decoder"
path  = "/content/repo"

if not os.path.exists(path):
    !git clone -q -b shuchi/dev https://{token}@github.com/{repo}.git {path}

%cd {path}
!nvidia-smi

/content/repo
Sun Nov 15 02:50:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------------

In [2]:
import pandas as pd, json

train = pd.read_parquet("data/splits/full_train.parquet")
val   = pd.read_parquet("data/splits/val.parquet")
labels = json.load(open("data/raw/label_names.json"))

print("train:", train.shape, "| val:", val.shape, "| labels:", len(labels))

train: (9618, 3) | val: (385, 3) | labels: 77


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)

print("Model loaded — parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("GPU memory used:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded — parameters: 1,543,714,304
GPU memory used: 3.09 GB


In [4]:
# same chat format we used for zero-shot, except fine tuning now

def format_example(row):
    """Turn one row into a chat the model learns to complete."""
    intent_name = labels[row["label"]]
    return {
        "messages": [
            {"role": "system", "content": "You classify banking customer messages. "
             "Reply with exactly one intent name and nothing else."},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": intent_name},
        ]
    }

# Format all training examples
train_chats = [format_example(row) for _, row in train.iterrows()]
val_chats   = [format_example(row) for _, row in val.iterrows()]

print("Training examples:", len(train_chats))
print("First example:")
for msg in train_chats[0]["messages"]:
    print(f"  {msg['role']}: {msg['content']}")

Training examples: 9618
First example:
  system: You classify banking customer messages. Reply with exactly one intent name and nothing else.
  user: I am still waiting on my card?
  assistant: card_arrival


In [7]:
!pip install -q trl

In [14]:
# set up time traner to run one epoch
import time
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_chats)

config = SFTConfig(
    output_dir="./timing_test",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    max_length=256,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=config,
)

print("Setup done. Starting training...")
print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f"\n--- TIMING RESULTS ---")
print(f"One epoch took: {elapsed/60:.1f} minutes")
print(f"GPU memory peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"GPU memory now:  {torch.cuda.memory_allocated()/1e9:.2f} GB")

Tokenizing train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/9618 [00:00<?, ? examples/s]

Setup done. Starting training...
GPU memory before training: 6.19 GB


Step,Training Loss
50,12165.716250
100,0.000000
150,0.000000
200,0.000000
250,0.000000
300,0.000000
350,0.000000
400,0.000000
450,0.000000
500,0.000000



--- TIMING RESULTS ---
One epoch took: 9.3 minutes
GPU memory peak: 13.85 GB
GPU memory now:  9.28 GB
